# Roman Urdu Hate Speech Detection
## Preprocessing Pipeline

**Team:** Muhammad Hunain Ashraf (463395) | Arfa Abdul Nasir (456634) | NUST

### Pipeline
1. Load all 4 datasets (RUHSOLD, HS-RU-20, RU-HSD-30K, RUT)
2. Harmonise labels → binary + RUHSOLD fine-grained
3. Drop duplicates & conflicts
4. Standard text cleaning
5. Spelling-variation augmentation (training set only)
6. Build vocabulary (min_freq=3)
7. Numericalize + pad (max_seq_len=64)
8. Load / align fastText embeddings
9. Train / Val / Test split
10. Save all artefacts

---

## 0. Imports & Constants

In [1]:
!pip install emoji gdown --quiet

import re, json, pickle, os, ast, requests, zipfile, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import emoji

tqdm.pandas()
os.makedirs('./data/processed', exist_ok=True)
os.makedirs('./artefacts',      exist_ok=True)

RANDOM_SEED  = 42
MIN_FREQ     = 3
MAX_SEQ_LEN  = 64      # increased from 50 to accommodate longer RUT comments
CHAR_MAX_LEN = 200     # char-level max length
UNK_TOKEN    = '<UNK>'
PAD_TOKEN    = '<PAD>'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('All imports OK.')
print(f'MIN_FREQ={MIN_FREQ} | MAX_SEQ_LEN={MAX_SEQ_LEN} | CHAR_MAX_LEN={CHAR_MAX_LEN}')

All imports OK.
MIN_FREQ=3 | MAX_SEQ_LEN=64 | CHAR_MAX_LEN=200


---
## 1. Load All Four Datasets

In [ ]:
#  Dataset 1: RUHSOLD (10,012 tweets, coarse + FINE-GRAINED labels) 

df_ruhsold = pd.read_csv('./data/rushold/RUSHOLD.csv')   

if 'label_fg' not in df_ruhsold.columns:
    df_ruhsold['label_fg'] = -1  # -1 = coarse-only sample

df_ruhsold = df_ruhsold[['tweet', 'label', 'label_fg']].dropna(subset=['tweet'])
df_ruhsold.columns = ['text', 'label_coarse', 'label_fg']
# coarse: 0→hate, 1→not_hate
df_ruhsold['label_binary'] = df_ruhsold['label_coarse'].apply(
    lambda x: 'hate' if float(x) == 0.0 else 'not_hate'
)
df_ruhsold['source'] = 'RUHSOLD'
print(f'RUHSOLD: {len(df_ruhsold):,} rows | fine-grained: {(df_ruhsold.label_fg != -1).sum():,}')

#  Dataset 2: HS-RU-20 (5,000 tweets) 
df_hsru = pd.read_excel('./data/HS-RU-20/Hate Speech Roman Urdu (HS-RU-20).xlsx')
df_hsru = df_hsru[['Sentence', 'Neutral (N) / Hostile (H)']].dropna()
df_hsru.columns = ['text', 'label_orig']
df_hsru['label_orig'] = df_hsru['label_orig'].str.strip()
df_hsru['label_binary'] = df_hsru['label_orig'].map({'H': 'hate', 'N': 'not_hate'})
df_hsru['label_fg'] = -1
df_hsru['source'] = 'HS-RU-20'
print(f'HS-RU-20: {len(df_hsru):,} rows')

#  Dataset 3: RU-HSD-30K (30,000 tweets)
df_30k = pd.read_csv('./data/RU-HSD-30K-main/final 30,000 dataset_romanurdu.csv', encoding='latin-1')
df_30k = df_30k[['tweets', 'label']].dropna()
df_30k.columns = ['text', 'label_orig']
df_30k['label_orig'] = df_30k['label_orig'].str.strip()
df_30k['label_binary'] = df_30k['label_orig'].map({'H': 'hate', 'N': 'not_hate'})
df_30k['label_fg'] = -1
df_30k['source'] = 'RU-HSD-30K'
print(f'RU-HSD-30K: {len(df_30k):,} rows')

# Dataset 4 : RUT — Roman Urdu Toxic Comments (72K)
df_rut = pd.read_parquet("hf://datasets/hafiz-hassaan-saeed/Roman-Urdu-Toxic-Corpus/RUT.parquet")
df_rut = df_rut[['Roman_Urdu', 'Toxic']].dropna()
df_rut.columns = ['text', 'label_rut']
df_rut['label_binary'] = df_rut['label_rut'].apply(
    lambda x: 'hate' if int(x) == 1 else 'not_hate'
)
df_rut['label_fg'] = -1
df_rut['source'] = 'RUT'
print(f'RUT: {len(df_rut):,} rows')

RUHSOLD: 9,212 rows | fine-grained: 9,212
HS-RU-20: 5,000 rows
RU-HSD-30K: 29,999 rows
RUT: 72,771 rows


In [ ]:
# Combine all four 
cols = ['text', 'label_binary', 'label_fg', 'source']
df = pd.concat([
    df_ruhsold[cols],
    df_hsru[cols],
    df_30k[cols],
    df_rut[cols]
], ignore_index=True)

df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)

print(f'Combined raw: {len(df):,} rows')
print(df['source'].value_counts())
print(df['label_binary'].value_counts())

Combined raw: 116,982 rows
source
RUT           72771
RU-HSD-30K    29999
RUHSOLD        9212
HS-RU-20       5000
Name: count, dtype: int64
label_binary
not_hate    83300
hate        33682
Name: count, dtype: int64


---
## 2. Drop Duplicates & Conflicts

In [4]:
before = len(df)
df = df.drop_duplicates(subset='text', keep='first').reset_index(drop=True)
print(f'Dropped {before - len(df):,} exact duplicates → {len(df):,} remaining')

# Find label conflicts (same text, different binary label)
df_full = pd.concat([
    df_ruhsold[cols], df_hsru[cols], df_30k[cols], df_rut[cols]
], ignore_index=True)
df_full['text'] = df_full['text'].astype(str).str.strip()

conflicts = (
    df_full.groupby('text')['label_binary'].nunique()
    .reset_index()
    .query('label_binary > 1')['text']
    .tolist()
)
df = df[~df['text'].isin(conflicts)].reset_index(drop=True)
print(f'Dropped {len(conflicts)} conflict texts → {len(df):,} remaining')

Dropped 17,477 exact duplicates → 99,505 remaining
Dropped 543 conflict texts → 98,962 remaining


---
## 3. Text Cleaning Pipeline

In [5]:
def clean_text(text: str) -> str:
    """Full cleaning pipeline — identical to v1 + extended."""
    # 1. Emoji → _emoji_name_ with spaces
    text = emoji.demojize(text, delimiters=(' _emoji_', '_ '))
    # 2. Strip # from hashtags, keep word (hashtag text preserved as a token)
    text = re.sub(r'#(\w+)', r'\1', text)
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # 4. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # 5. Lowercase
    text = text.lower()
    # 6. Collapse 3+ repeated chars → 2
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 7. Remove punctuation (keep apostrophes + underscores for emoji tokens)
    text = re.sub(r"[^\w\s'_]", ' ', text)
    # 8. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

tqdm.pandas(desc='Cleaning')
df['text_clean'] = df['text'].progress_apply(clean_text)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)
print(f'After cleaning: {len(df):,} rows')

Cleaning:   0%|          | 0/98962 [00:00<?, ?it/s]

After cleaning: 98,885 rows


---
## 4. Train / Val / Test Split (before augmentation — no leakage)

In [6]:
from sklearn.model_selection import train_test_split

# Stratify by label_binary
df_train_val, df_test = train_test_split(
    df, test_size=0.15, stratify=df['label_binary'], random_state=RANDOM_SEED
)
df_train, df_val = train_test_split(
    df_train_val, test_size=0.15/(1-0.15),
    stratify=df_train_val['label_binary'], random_state=RANDOM_SEED
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')
print('Train label dist:', df_train['label_binary'].value_counts().to_dict())

# Leakage check
assert len(set(df_train['text']) & set(df_test['text']))  == 0, 'Train-Test leak!'
assert len(set(df_train['text']) & set(df_val['text']))   == 0, 'Train-Val leak!'
assert len(set(df_val['text'])   & set(df_test['text']))  == 0, 'Val-Test leak!'
print('Zero leakage confirmed ✓')

Train: 69,219 | Val: 14,833 | Test: 14,833
Train label dist: {'not_hate': 48856, 'hate': 20363}
Zero leakage confirmed ✓


---
## 5. Spelling-Variation Augmentation (Training Set Only)

Generates additional training samples by randomly applying common Roman Urdu
spelling variant substitutions. This directly addresses the high OOV rate caused
by orthographic inconsistency - the DL models (especially the char-CNN branch)
can exploit variant forms that TF-IDF treats as completely different tokens.

Augmentation rate: 30% of hate class samples doubled.

In [ ]:
# Common Roman Urdu orthographic variant substitutions
VARIANT_RULES = [
    # (pattern, list_of_replacements) - one chosen at random
    (r'\bhai\b',   ['ha', 'hy', 'hey']),
    (r'\bha\b',    ['hai', 'hy']),
    (r'\bhy\b',    ['hai', 'ha']),
    (r'\bnahi\b',  ['ni', 'nai', 'nahin', 'nhi']),
    (r'\bni\b',    ['nahi', 'nai', 'nhi']),
    (r'\bnai\b',   ['nahi', 'ni', 'nhi']),
    (r'\bmujhe\b', ['mujhay', 'mjhe', 'mujhai', 'mujhy']),
    (r'\bhon\b',   ['hun', 'hoon']),
    (r'\bkaro\b',  ['kro', 'karo']),
    (r'\bap\b',    ['aap']),
    (r'\baap\b',   ['ap']),
    (r'\bkya\b',   ['kia', 'kya']),
    (r'\bkia\b',   ['kya', 'kia']),
    (r'\bko\b',    ['ku', 'ko']),
    (r'\bky\b',    ['kyu', 'kyun', 'kyunke']),
]

def augment_text(text: str, n_rules: int = 2) -> str:
    """Apply n_rules random substitutions to create a spelling variant."""
    rules = random.sample(VARIANT_RULES, min(n_rules, len(VARIANT_RULES)))
    for pattern, replacements in rules:
        repl = random.choice(replacements)
        text = re.sub(pattern, repl, text)
    return text

random.seed(RANDOM_SEED)
# Augment 30% of training samples (both classes, keep balance)
aug_mask = df_train.sample(frac=0.30, random_state=RANDOM_SEED).index
df_aug = df_train.loc[aug_mask].copy()
df_aug['text_clean'] = df_aug['text_clean'].apply(augment_text)
df_aug['is_augmented'] = True
df_train['is_augmented'] = False

df_train_aug = pd.concat([df_train, df_aug], ignore_index=True).sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f'Train after augmentation: {len(df_train_aug):,} (was {len(df_train):,})')
print(f'Augmented samples added: {len(df_aug):,}')

Train after augmentation: 89,985 (was 69,219)
Augmented samples added: 20,766


---
## 6. Tokenisation & Vocabulary

In [8]:
# Build vocab on TRAINING set only (augmented)
all_train_tokens = [
    tok for text in df_train_aug['text_clean']
    for tok in text.split()
]
freq = Counter(all_train_tokens)
print(f'Unique tokens (train): {len(freq):,}')

vocab_tokens = [tok for tok, cnt in freq.items() if cnt >= MIN_FREQ]
print(f'Vocab tokens (freq≥{MIN_FREQ}): {len(vocab_tokens):,}')

word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for tok in sorted(vocab_tokens):
    word2idx[tok] = len(word2idx)
idx2word  = {v: k for k, v in word2idx.items()}
VOCAB_SIZE = len(word2idx)
PAD_IDX    = 0
UNK_IDX    = 1
print(f'Final vocab size: {VOCAB_SIZE:,}')

Unique tokens (train): 89,087
Vocab tokens (freq≥3): 34,041
Final vocab size: 34,043


---
## 6b. UNK Ratio Inspection
Flag training tweets where >50% of tokens are `<UNK>` after vocabulary filtering.
This diagnostic surfaces tweets that are effectively destroyed by the `min_freq` cutoff -
useful for catching tokenisation problems or domain-shift issues in the RUT corpus.

In [ ]:
def compute_unk_ratio(text: str) -> float:
    tokens = text.split()
    if len(tokens) == 0:
        return 1.0
    return sum(1 for t in tokens if word2idx.get(t, UNK_IDX) == UNK_IDX) / len(tokens)

# Compute on the full (pre-augmentation) training set for a clean signal
df_train['unk_ratio'] = df_train['text_clean'].apply(compute_unk_ratio)

print('UNK ratio distribution (train):')
print(df_train['unk_ratio'].describe().round(3).to_string())
print()
for thresh in [0.3, 0.5, 0.7, 1.0]:
    n = (df_train['unk_ratio'] >= thresh).sum()
    print(f'  unk_ratio >= {thresh}: {n:,} tweets ({n/len(df_train)*100:.2f}%)')

# Show worst offenders — are they real content or noise?
high_unk = df_train[df_train['unk_ratio'] >= 0.5].sort_values('unk_ratio', ascending=False)
print(f'\nTweets with unk_ratio >= 0.5: {len(high_unk):,}')
print('\nSource breakdown among high-UNK tweets:')
print(high_unk['source'].value_counts().to_string())
print('\nLabel distribution among high-UNK tweets:')
print(high_unk['label_binary'].value_counts().to_string())
print('\nSamples (worst first):')
display(
    high_unk[['text', 'text_clean', 'label_binary', 'source', 'unk_ratio']].head(20)
)

before = len(df_train)
df_train = df_train[df_train['unk_ratio'] < 0.5].reset_index(drop=True)
print(f'Dropped {before - len(df_train)} high-UNK tweets. Remaining: {len(df_train):,}')

UNK ratio distribution (train):
count    69219.000
mean         0.056
std          0.110
min          0.000
25%          0.000
50%          0.000
75%          0.077
max          1.000

  unk_ratio >= 0.3: 2,031 tweets (2.93%)
  unk_ratio >= 0.5: 960 tweets (1.39%)
  unk_ratio >= 0.7: 378 tweets (0.55%)
  unk_ratio >= 1.0: 360 tweets (0.52%)

Tweets with unk_ratio >= 0.5: 960

Source breakdown among high-UNK tweets:
source
RUT           705
RU-HSD-30K    236
RUHSOLD        19

Label distribution among high-UNK tweets:
label_binary
not_hate    619
hate        341

Samples (worst first):


,text,text_clean,label_binary,source,unk_ratio
202,ghalllat,ghallat,not_hate,RU-HSD-30K,1.0
67733,Helwwwwwww,helww,not_hate,RUT,1.0
67660,Tagda.....,tagda,not_hate,RUT,1.0
67632,tattibhai gandhiyoutuber,tattibhai gandhiyoutuber,hate,RU-HSD-30K,1.0
855,Yarl jajb,yarl jajb,not_hate,RUT,1.0
2130,Aurauraurrrrrrln,aurauraurrln,not_hate,RUT,1.0
1990,#WeStandWithPakArmy,westandwithpakarmy,not_hate,RUT,1.0
1965,Jakas,jakas,hate,RUT,1.0
1961,Alarabia,alarabia,not_hate,RUT,1.0
1887,#GDBakhsiKiMaKaBhosda,gdbakhsikimakabhosda,hate,RUT,1.0


Dropped 960 high-UNK tweets. Remaining: 68,259


In [11]:
# Character vocabulary
CHARS = list("abcdefghijklmnopqrstuvwxyz0123456789 '-_")
char2idx = {'<PAD>': 0, '<UNK>': 1}
for c in CHARS:
    char2idx[c] = len(char2idx)
CHAR_VOCAB_SIZE = len(char2idx)
CHAR_PAD_IDX    = 0
print(f'Char vocab size: {CHAR_VOCAB_SIZE}')

Char vocab size: 42


In [12]:
# Label maps — binary
label2idx = {'not_hate': 0, 'hate': 1}
idx2label = {0: 'not_hate', 1: 'hate'}

# Fine-grained label map (for multi-task auxiliary head)
# -1 = unknown (binary-only sample, masked during aux loss)
FINE_LABEL_MAP = {
    -1: -1,  # masked
     0:  0,  # Abusive/Offensive
     1:  1,  # Normal
     2:  2,  # Religious Hate
     3:  3,  # Sexism
     4:  4,  # Profane/Untargeted
}
NUM_FINE_CLASSES = 5
print('Label maps created. Fine-grained classes:', NUM_FINE_CLASSES)

Label maps created. Fine-grained classes: 5


---
## 7. Numericalize, Char-encode, Pad/Truncate

In [ ]:
def numericalize(text: str) -> list:
    tokens = text.split()
    ids = [word2idx.get(tok, UNK_IDX) for tok in tokens]
    # Pad/truncate
    if len(ids) < MAX_SEQ_LEN:
        ids += [PAD_IDX] * (MAX_SEQ_LEN - len(ids))
    else:
        ids = ids[:MAX_SEQ_LEN]
    return ids

def char_encode(text: str) -> list:
    ids = [char2idx.get(c, 1) for c in text.lower()]
    if len(ids) < CHAR_MAX_LEN:
        ids += [CHAR_PAD_IDX] * (CHAR_MAX_LEN - len(ids))
    else:
        ids = ids[:CHAR_MAX_LEN]
    return ids

for split_df, name in [(df_train_aug, 'train_aug'), (df_val, 'val'), (df_test, 'test')]:
    tqdm.pandas(desc=f'Numericalizing {name}')
    split_df['input_ids']  = split_df['text_clean'].progress_apply(numericalize)
    split_df['char_ids']   = split_df['text_clean'].progress_apply(char_encode)
    split_df['label_idx']  = split_df['label_binary'].map(label2idx)
    split_df['label_fg_idx'] = split_df['label_fg'].apply(
        lambda x: FINE_LABEL_MAP.get(int(x), -1)
    )

print('Numericalization complete ')

Numericalizing train_aug:   0%|          | 0/89985 [00:00<?, ?it/s]

Numericalizing train_aug:   0%|          | 0/89985 [00:00<?, ?it/s]

Numericalizing val:   0%|          | 0/14833 [00:00<?, ?it/s]

Numericalizing val:   0%|          | 0/14833 [00:00<?, ?it/s]

Numericalizing test:   0%|          | 0/14833 [00:00<?, ?it/s]

Numericalizing test:   0%|          | 0/14833 [00:00<?, ?it/s]

Numericalization complete ✓


---
## 8. Download & Align fastText Embeddings (EMNLP 2020)

The EMNLP 2020 paper trained fastText on **4.7M Roman Urdu tweets** and
released pre-trained vectors. These replace random embedding initialisation,
giving DL models a head-start that TF-IDF cannot leverage.

In [ ]:
import os

# Step 1: Download fastText vectors 
# Source:  https://aclanthology.org/2020.emnlp-main.197/

FASTTEXT_URL = (
    'https://drive.google.com/uc?export=download'
    '&id=1sn8GbkPSfMmklcRp1xUQlTuUoNPMBcIC'  # Roman Urdu fastText (dim=100)
)
FASTTEXT_PATH = './artefacts/ft_sg.vec'

if not os.path.exists(FASTTEXT_PATH):
    print('Downloading fastText vectors (this may take a minute)...')
    try:
        import gdown
        gdown.download(FASTTEXT_URL, FASTTEXT_PATH, quiet=False)
    except Exception as e:
        print(f'Auto-download failed ({e}).')
        print('MANUAL DOWNLOAD INSTRUCTIONS:')
        print('1. Go to: https://aclanthology.org/2020.emnlp-main.197/')
        print('2. Find supplementary materials / resources link')
        print('3. Download the .vec or .bin file for Roman Urdu fastText')
        print(f'4. Place it at: {FASTTEXT_PATH}')
        print()
        print('Alternative: Use the Kaggle companion dataset:')
        print('  https://www.kaggle.com/datasets/devzohaib/final-ruhsp-experiments')
        print()
        FASTTEXT_PATH = None
else:
    print(f'fastText vectors already at {FASTTEXT_PATH}')

fastText vectors already at ./artefacts/ft_sg.vec


In [16]:
def load_fasttext_vectors(vec_path: str, word2idx: dict, embed_dim: int = 100):
    """
    Load fastText .vec file and build an embedding matrix aligned to word2idx.
    Tokens not found in fastText are left as random normal vectors.

    Returns:
        embed_matrix: np.ndarray of shape (vocab_size, embed_dim)
        coverage: fraction of vocab found in fastText
    """
    vocab_size = len(word2idx)
    embed_matrix = np.random.normal(scale=0.1, size=(vocab_size, embed_dim)).astype(np.float32)
    embed_matrix[word2idx.get('<PAD>', 0)] = 0.0  # PAD = zero vector

    # Make lookup robust to casing differences.
    word2idx_lower = {str(w).lower(): i for w, i in word2idx.items()}

    hits = 0
    bad_lines = 0

    with open(vec_path, 'r', encoding='utf-8', errors='ignore') as f:
        header = f.readline().strip()  # usually: <num_words> <dim>

        # Detect common "downloaded HTML instead of .vec" failure mode.
        if '<html' in header.lower() or '<!doctype' in header.lower():
            raise ValueError(
                'FASTTEXT_PATH does not point to a valid .vec file (HTML detected). Re-download the vectors.'
            )

        # Validate header if present (many .vec files use this format).
        header_parts = header.split()
        if len(header_parts) >= 2 and header_parts[1].isdigit():
            file_dim = int(header_parts[1])
            if file_dim != embed_dim:
                raise ValueError(f'Embedding dim mismatch: file has {file_dim}, expected {embed_dim}')

        for line in f:
            parts = line.rstrip().split()  # split on any whitespace
            if len(parts) < embed_dim + 1:
                bad_lines += 1
                continue

            word = parts[0].lower()
            idx = word2idx_lower.get(word, None)
            if idx is None:
                continue

            try:
                vec = np.asarray(parts[1:1 + embed_dim], dtype=np.float32)
            except ValueError:
                bad_lines += 1
                continue

            if vec.shape[0] == embed_dim:
                embed_matrix[idx] = vec
                hits += 1

    coverage = hits / max(vocab_size, 1)
    print(f'fastText coverage: {hits}/{vocab_size} ({coverage:.1%})')
    if bad_lines:
        print(f'Skipped malformed vector lines: {bad_lines}')

    return embed_matrix, coverage


def infer_fasttext_dim(vec_path: str):
    """Infer embedding dim from .vec header '<num_words> <dim>' if available."""
    try:
        with open(vec_path, 'r', encoding='utf-8', errors='ignore') as f:
            header = f.readline().strip().split()
        if len(header) >= 2 and header[1].isdigit():
            return int(header[1])
    except Exception:
        pass
    return None


DEFAULT_EMBED_DIM = 300
embed_matrix = None

if FASTTEXT_PATH and os.path.exists(FASTTEXT_PATH):
    inferred_dim = infer_fasttext_dim(FASTTEXT_PATH)
    EMBED_DIM = inferred_dim if inferred_dim is not None else DEFAULT_EMBED_DIM
    FASTTEXT_EMBED_DIM = EMBED_DIM

    if inferred_dim is not None and inferred_dim != DEFAULT_EMBED_DIM:
        print(f'Info: fastText file dim is {inferred_dim}; overriding default {DEFAULT_EMBED_DIM}.')

    try:
        embed_matrix, coverage = load_fasttext_vectors(FASTTEXT_PATH, word2idx, EMBED_DIM)

        # 0% (or near-zero) usually means a wrong/corrupt file, not true OOV.
        if coverage <= 0.0001:
            raise RuntimeError(
                'fastText coverage is ~0%. File is likely invalid or tokenization is mismatched.'
            )

        np.save('./artefacts/embed_matrix.npy', embed_matrix)
        print(f'Embedding matrix saved: {embed_matrix.shape}')
    except Exception as e:
        print(f'fastText loading failed: {e}')
        print('Falling back to random embedding matrix.')
        vocab_size = len(word2idx)
        embed_matrix = np.random.normal(scale=0.1, size=(vocab_size, EMBED_DIM)).astype(np.float32)
        embed_matrix[word2idx.get('<PAD>', 0)] = 0.0
        np.save('./artefacts/embed_matrix.npy', embed_matrix)
        print(f'Random fallback embedding matrix saved: {embed_matrix.shape}')
else:
    EMBED_DIM = DEFAULT_EMBED_DIM
    FASTTEXT_EMBED_DIM = EMBED_DIM
    print('fastText not available — models will use random embedding init.')
    vocab_size = len(word2idx)
    embed_matrix = np.random.normal(scale=0.1, size=(vocab_size, EMBED_DIM)).astype(np.float32)
    embed_matrix[word2idx.get('<PAD>', 0)] = 0.0
    np.save('./artefacts/embed_matrix.npy', embed_matrix)
    print(f'Random fallback embedding matrix saved: {embed_matrix.shape}')

fastText coverage: 14420/16176 (89.1%)
Skipped malformed vector lines: 2
Embedding matrix saved: (16176, 300)


---
## 9. Save All Artefacts

In [ ]:
#  Vocabulary 
with open('./artefacts/word2idx.json', 'w', encoding='utf-8') as f:
    json.dump(word2idx, f, ensure_ascii=False)
with open('./artefacts/idx2word.json', 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in idx2word.items()}, f, ensure_ascii=False)
with open('./artefacts/char2idx.json', 'w') as f:
    json.dump(char2idx, f)
with open('./artefacts/label2idx.json', 'w') as f:
    json.dump(label2idx, f)
with open('./artefacts/idx2label.json', 'w') as f:
    json.dump({str(k): v for k, v in idx2label.items()}, f)

# Config 
config = {
    'vocab_size':        VOCAB_SIZE,
    'char_vocab_size':   CHAR_VOCAB_SIZE,
    'pad_idx':           PAD_IDX,
    'unk_idx':           UNK_IDX,
    'char_pad_idx':      CHAR_PAD_IDX,
    'max_seq_len':       MAX_SEQ_LEN,
    'char_max_len':      CHAR_MAX_LEN,
    'embed_dim':         EMBED_DIM,
    'min_freq':          MIN_FREQ,
    'num_classes':       2,
    'num_fine_classes':  NUM_FINE_CLASSES,
    'random_seed':       RANDOM_SEED,
    'train_size':        len(df_train_aug),
    'val_size':          len(df_val),
    'test_size':         len(df_test),
    'fasttext_dim':      FASTTEXT_EMBED_DIM,
}
with open('./artefacts/config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config saved.')

Config saved.


In [ ]:
#  Numpy arrays 
def to_numpy(split_df):
    ids = split_df['input_ids'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    char_ids = split_df['char_ids'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    X      = np.array(ids.tolist(),      dtype=np.int64)
    X_char = np.array(char_ids.tolist(), dtype=np.int64)
    y      = np.array(split_df['label_idx'].tolist(),    dtype=np.int64)
    y_fg   = np.array(split_df['label_fg_idx'].tolist(), dtype=np.int64)
    return X, X_char, y, y_fg

X_train, X_train_char, y_train, y_train_fg = to_numpy(df_train_aug)
X_val,   X_val_char,   y_val,   y_val_fg   = to_numpy(df_val)
X_test,  X_test_char,  y_test,  y_test_fg  = to_numpy(df_test)

np.save('./artefacts/X_train.npy',      X_train)
np.save('./artefacts/X_train_char.npy', X_train_char)
np.save('./artefacts/y_train.npy',      y_train)
np.save('./artefacts/y_train_fg.npy',   y_train_fg)

np.save('./artefacts/X_val.npy',        X_val)
np.save('./artefacts/X_val_char.npy',   X_val_char)
np.save('./artefacts/y_val.npy',        y_val)
np.save('./artefacts/y_val_fg.npy',     y_val_fg)

np.save('./artefacts/X_test.npy',       X_test)
np.save('./artefacts/X_test_char.npy',  X_test_char)
np.save('./artefacts/y_test.npy',       y_test)
np.save('./artefacts/y_test_fg.npy',    y_test_fg)

print('Numpy arrays saved:')
print(f'  X_train:      {X_train.shape}  X_train_char: {X_train_char.shape}')
print(f'  X_val:        {X_val.shape}    X_val_char:   {X_val_char.shape}')
print(f'  X_test:       {X_test.shape}   X_test_char:  {X_test_char.shape}')

Numpy arrays saved:
  X_train:      (89985, 64)  X_train_char: (89985, 200)
  X_val:        (14833, 64)    X_val_char:   (14833, 200)
  X_test:       (14833, 64)   X_test_char:  (14833, 200)


In [ ]:
# Save CSV splits 
cols_save = ['text', 'text_clean', 'label_binary', 'label_idx',
             'label_fg', 'label_fg_idx', 'source']
df_train_aug[cols_save].to_csv('./data/processed/train.csv', index=False)
df_val[cols_save].to_csv('./data/processed/val.csv',         index=False)
df_test[cols_save].to_csv('./data/processed/test.csv',       index=False)
print('CSV splits saved.')

print('\n' + '='*55)
print('PREPROCESSING v2 COMPLETE — SUMMARY')
print('='*55)
print(f'Datasets used            : RUHSOLD + HS-RU-20 + RU-HSD-30K + RUT')
print(f'Train (with aug)         : {len(df_train_aug):,}')
print(f'Val                      : {len(df_val):,}')
print(f'Test                     : {len(df_test):,}')
print(f'Vocab size               : {VOCAB_SIZE:,}')
print(f'Char vocab size          : {CHAR_VOCAB_SIZE}')
print(f'Seq len (word / char)    : {MAX_SEQ_LEN} / {CHAR_MAX_LEN}')
print(f'Embed dim (fastText)     : {EMBED_DIM}')
print(f'Fine-grained aux classes : {NUM_FINE_CLASSES} (masked where unavailable)')
print('\nNext: Notebook 03 — Upgraded Model Training')

CSV splits saved.

PREPROCESSING v2 COMPLETE — SUMMARY
Datasets used            : RUHSOLD + HS-RU-20 + RU-HSD-30K + RUT
Train (with aug)         : 89,985
Val                      : 14,833
Test                     : 14,833
Vocab size               : 34,043
Char vocab size          : 42
Seq len (word / char)    : 64 / 200
Embed dim (fastText)     : 300
Fine-grained aux classes : 5 (masked where unavailable)

Next: Notebook 03 — Upgraded Model Training
